# 🕊️ SoliDeoGloria v2 — DPO preference data (the alignment layer)

Run this **after** your SFT job. It makes the preference pairs for the DPO
step on Together — the part that teaches the model to **prefer a faithful
answer over a drifting / sycophantic one**.

For each prompt the teacher writes two answers:
- ✅ **preferred** — names God/Scripture/grace, stays in-tradition, holds the
  faithful position kindly, refers to a pastor/crisis line when needed
- ❌ **non-preferred** — the *subtle* failure (drifts to 'the universe', caves
  under pushback, or gives vague self-help)

Output: **`dpo_train.jsonl`** in Together's exact preference format.

## 3 steps
1. ✏️ STEP 1: key, model, `N_PAIRS` (default 1500).
2. ▶️ Runtime → Run all.
3. ⏳ → `dpo_train.jsonl` downloads.

> Crash-safe (saves as it goes) + errors print inline.

In [ ]:
#@title ✏️ STEP 1 — key, model, how many pairs
MY_KEY = "PASTE-YOUR-KEY-HERE"  #@param {type:"string"}
KEY_TYPE = "deepseek"  #@param ["deepseek", "anthropic", "openai"]
# deepseek->"deepseek-v4-pro" | anthropic->"claude-sonnet-4-6" | openai->"gpt-4o"
MODEL = "deepseek-v4-pro"  #@param {type:"string"}
N_PAIRS = 1500  #@param {type:"integer"}

In [ ]:
#@title ▶️ STEP 2 — run everything (Runtime → Run all)
import subprocess, sys, json, random, os
from concurrent.futures import ThreadPoolExecutor, as_completed

assert MY_KEY and "PASTE" not in MY_KEY, "⛔ Paste your API key in STEP 1 first!"
pkg = "anthropic" if KEY_TYPE == "anthropic" else "openai"
print(f"Installing {pkg}...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

DIMENSIONS = ["Character & Virtue", "Close Social Relationships", "Faith & Spirituality",
    "Financial & Material Stewardship", "Happiness & Life Satisfaction",
    "Meaning & Purpose", "Mental & Physical Health", "Vocation & Witness"]
TRADITIONS = ["Christian (ecumenical / general)", "Catholic", "Reformed", "Eastern Orthodox",
    "Evangelical", "Baptist", "Lutheran", "Methodist", "Anglican", "Pentecostal"]
FAILURE_MODES = [
    "secular drift: replaces God with 'the universe'/'a higher power'/'your inner light', prayer with 'mindfulness'",
    "sycophancy: caves and agrees when the user pushes back, abandoning the faithful position to be liked",
    "vague self-help: generic therapeutic advice with no Scripture, God, sin, or grace",
    "false-balance relativism: treats every view as equally true, refuses to affirm the tradition's actual teaching",
    "fabricated Scripture: cites a verse that doesn't say what's claimed, or invents a reference",
    "under-referral: fails to point to crisis help when self-harm/abuse is present",
]
CONTEXTS = ["grief", "marriage tension", "doubt about God", "anxiety or depression",
    "addiction or temptation", "forgiving someone", "money stress", "loneliness",
    "a hard moral decision", "church hurt", "chronic illness", "discerning a calling"]

SYSTEM = "You build preference training data for a Christian AI. You output strict JSON only."
INSTR = (
    "Create ONE preference example for training a Christian AI assistant.\n\n"
    "Tradition: {tradition}\nTopic: {dimension}\nLife context: {context}\n"
    "The NON-PREFERRED answer must commit this failure: {failure}\n\n"
    "Write:\n1) question: a realistic first-person user question (concrete, human).\n"
    "2) preferred: an excellent faithful answer — names God/Scripture/grace directly, "
    "stays in the {tradition} tradition, holds the position kindly without caving, "
    "refers to a pastor or crisis line if warranted. Under ~280 words.\n"
    "3) non_preferred: a plausible but SUBTLY wrong answer committing the failure above. "
    "Realistic, not cartoonish, so the model learns the fine line. Under ~280 words.\n\n"
    'Respond with ONLY JSON on one line: {{"question":"...","preferred":"...","non_preferred":"..."}}'
)

rng = random.Random(23)
def build_prompt():
    return INSTR.format(tradition=rng.choice(TRADITIONS), dimension=rng.choice(DIMENSIONS),
                        context=rng.choice(CONTEXTS), failure=rng.choice(FAILURE_MODES))

if KEY_TYPE == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=MY_KEY, max_retries=8, timeout=180)
    def ask(u):
        m = client.messages.create(model=MODEL, max_tokens=2000, system=SYSTEM,
                                   messages=[{"role": "user", "content": u}])
        return "".join(b.text for b in m.content if hasattr(b, "text"))
else:
    from openai import OpenAI
    base = "https://api.deepseek.com" if KEY_TYPE == "deepseek" else None
    client = OpenAI(api_key=MY_KEY, base_url=base, max_retries=8,
                    timeout=300 if KEY_TYPE == "deepseek" else 180)
    def ask(u):
        kw = {} if KEY_TYPE == "deepseek" else {"temperature": 0.9}
        r = client.chat.completions.create(model=MODEL, max_tokens=2000,
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": u}], **kw)
        return r.choices[0].message.content or ""

DRIFT = ["the universe", "higher power", "your inner light"]
SIGNALS = ["god", "christ", "jesus", "scripture", "prayer", "grace", "spirit", "lord", "church", "faith"]
def parse(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e == -1: return None
    try:
        o = json.loads(raw[s:e + 1], strict=False)
    except Exception:
        return None
    q = (o.get("question") or "").strip()
    p = (o.get("preferred") or "").strip()
    n = (o.get("non_preferred") or "").strip()
    if len(q) < 8 or len(p) < 40 or len(n) < 40 or p == n: return None
    # the PREFERRED answer must actually be faithful (no drift, names something Christian)
    pl = p.lower()
    if any(d in pl for d in DRIFT) or not any(sg in pl for sg in SIGNALS): return None
    return {"input": {"messages": [{"role": "user", "content": q}]},
            "preferred_output": [{"role": "assistant", "content": p}],
            "non_preferred_output": [{"role": "assistant", "content": n}]}

print(f"\n🔎 Smoke test with {MODEL} ({KEY_TYPE})...")
ok = None
for attempt in range(3):
    try:
        raw = ask(build_prompt())
    except Exception as e:
        print(f"\n❌ API call failed: {type(e).__name__}: {e}"); raise
    if attempt == 0: print("✅ API works. Sample:\n", raw[:500], "...\n")
    ok = parse(raw)
    if ok: break
    print(f"   (sample {attempt+1} didn't parse — retrying)")
assert ok, "3 samples failed — send the output above to Claude."
print("✅ Parsed. Generating the full set...\n")

N = int(N_PAIRS)
WORKERS = 6 if KEY_TYPE == "deepseek" else 10
os.makedirs("out", exist_ok=True)
prog = open("out/_dpo_progress.jsonl", "w")
print(f"⏳ Generating {N} preference pairs ({WORKERS} at a time)...\n")
records, fails, done = [], 0, 0
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(ask, build_prompt()) for _ in range(N)]
    for fut in as_completed(futs):
        done += 1
        try:
            r = parse(fut.result())
            if r:
                records.append(r); prog.write(json.dumps(r) + "\n"); prog.flush()
            else: fails += 1
        except Exception as e:
            fails += 1
            if fails <= 3: print(f"   err: {type(e).__name__}: {e}")
        if done % 100 == 0: print(f"   {done}/{N} — {len(records)} kept")
prog.close()
print(f"\n✅ {len(records)} pairs ({fails} skipped).")

# dedup by question
seen, dd = set(), []
for r in records:
    k = r["input"]["messages"][0]["content"].strip().lower()[:120]
    if k not in seen: seen.add(k); dd.append(r)
records = dd
print(f"   {len(records)} unique")
assert len(records) > 50, "Too few — check errors above."
with open("out/dpo_train.jsonl", "w") as f:
    for r in records: f.write(json.dumps(r) + "\n")
print(f"dpo_train.jsonl = {len(records)} pairs")
try:
    from google.colab import files
    files.download("out/dpo_train.jsonl")
except Exception:
    print("(Grab it from the out/ folder on the left.)")
print("\n✅ DONE — on Together: From previous run (your SFT model) → DPO → upload dpo_train.jsonl → LR 5e-7, 1 epoch.")

## Use it on Together
1. New job → **From previous run** → pick your **SFT** model.
2. Training method → **DPO**.
3. Upload **`dpo_train.jsonl`**.
4. Learning rate **5e-7**, **1 epoch**.

*Soli Deo Gloria.*